In [1]:
import psycopg2
from dotenv import load_dotenv
import os

In [2]:
# Load environment variables from .env
load_dotenv()

# Fetch variables
USER = os.getenv("user")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")

# Connect to the database
try:
    connection = psycopg2.connect(
        user=USER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME
    )
    print("Connection successful!")
    
    # Create a cursor to execute SQL queries
    cursor = connection.cursor()
    
    # Example query
    cursor.execute("SELECT NOW();")
    result = cursor.fetchone()
    print("Current Time:", result)

    # Close the cursor and connection
    cursor.close()
    connection.close()
    print("Connection closed.")

except Exception as e:
    print(f"Failed to connect: {e}")

Connection successful!
Current Time: (datetime.datetime(2026, 3, 14, 15, 1, 8, 276704, tzinfo=datetime.timezone.utc),)
Connection closed.


In [3]:
import anthropic
import pandas as pd

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-4-6")
print("Anthropic listo:", CLAUDE_MODEL)

Anthropic listo: claude-sonnet-4-6


In [4]:
import json
from psycopg2.extras import execute_values

SCHEMA_CONTEXT = """
Tabla: haciendas
Granularidad: un registro = una hacienda + un mes.
Cobertura temporal: enero 2020 a junio 2025.

Columnas:
- FECHA (date): fecha del registro (tipo DATE en PostgreSQL, formato YYYY-MM-DD)
- Semana (int): número de semana del año
- Zona (text): región geográfica
- Unidad (text): código identificador de la hacienda
- Nombre_Unidad (text): nombre de la hacienda
- Real (numeric): indicador de rendimiento/ratio de producción real
- Costo_Ha (numeric): costo total acumulado por hectárea
- Atencion_Plantacion (numeric): costos de mantenimiento del cultivo
- C_Riego (numeric): costo total de riego
- C_Mano_Obra_Riego (numeric): costo de personal para riego
- C_Mantenimiento_Riego (numeric): costo de reparaciones de infraestructura de riego
- C_Combustible (numeric): costo de combustible
- C_Control_Sigatoca (numeric): costo del programa contra la Sigatoka
- C_Aplicacion_Aerea (numeric): costo de fumigación aérea
- C_Deshoje (numeric): costo de deshoje
- C_Costos_Productos (numeric): costo en insumos químicos y fertilizantes
- C_Fertilizacion (numeric): costo total de fertilización
- C_Sacos_Fert (numeric): costo de sacos de fertilizante
- C_ManodeObra_Fert (numeric): costo de aplicación de fertilizante
- C_Transporte_Fert (numeric): costo de transporte de fertilizante
- C_Administracion_Hacienda (numeric): costos administrativos
- Sueldos (numeric): nómina de empleados fijos
- Servicios_Basicos (numeric): pagos de luz, agua y otros servicios
- C_Empaque_Fijo (numeric): costos fijos de empaque
- Mantenimiento_Empacadora (numeric): costos de mantenimiento de empacadora
- Mantenimiento_Equipo (numeric): costo de mantenimiento de equipo
- C_Logistica (numeric): costo total de logística
- Transporte (numeric): gasto en fletes y acarreo
- Materiales (numeric): inversión en insumos de empaque
- Reclasificaciones_Transporte (numeric): ajustes contables sobre transporte
- Reclasificaciones_Materiales (numeric): ajustes contables sobre materiales
- C_Empaque_Variable (numeric): costos variables de empaque
- C_Cosecha (numeric): costo de cosecha
- C_Transporte (numeric): otros costos de transporte
- C_Depreciaciones (numeric): depreciación de activos fijos
- Total_Cajas (numeric): volumen total de cajas producidas
- Total_Hectareas (numeric): superficie productiva en hectáreas
- Racimo_Rechazado (numeric): cantidad de fruta rechazada
- Total_Peso_Caja (numeric): peso total de las cajas
- Promedio_Peso_Caja (numeric): peso promedio por caja
- Tipo_Suelo (text): clasificación del terreno
- Incidencia_Sigatoka (numeric): nivel de presencia de la plaga
- Temperatura_C (numeric): temperatura media en °C
- Precipitacion_mm (numeric): lluvia acumulada en mm
- Evotranspiracion (numeric): tasa de evapotranspiración
- Humedad (numeric): porcentaje de humedad relativa
- Ausentismo_Agricola (numeric): total de inasistencias del personal
- Ausentismo_Justificado_Agricola (numeric): inasistencias justificadas
- Ausentismo_Injustificado_Agricola (numeric): inasistencias injustificadas
- RotPerson_Salida_Todos_Motivos_Agricola (numeric): índice de rotación de personal
- Pago_Labor_Persona (numeric): indicador de pago por jornada
- Pago_Por_Cuenta (numeric): indicador de pagos por cuenta
- Vacante_Labor (numeric): puestos de trabajo vacantes
"""

COLUMNAS_VALIDAS = [
    line.split("(")[0].replace("-", "").strip()
    for line in SCHEMA_CONTEXT.splitlines()
    if line.strip().startswith("-")
]
COLUMNAS_VALIDAS_STR = ", ".join(f'"{c}"' for c in COLUMNAS_VALIDAS)

print(f"Schema cargado — {len(COLUMNAS_VALIDAS)} columnas")

Schema cargado — 53 columnas


In [5]:
def _llm(prompt: str) -> str:
    message = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=4096,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text.strip()

print("_llm listo")

_llm listo


In [6]:
def _limpiar_sql(texto: str) -> str:
    texto = texto.strip()
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("sql"):
            texto = texto[3:]
    return texto.strip()

print("_limpiar_sql listo")

_limpiar_sql listo


In [7]:
def _conectar():
    return psycopg2.connect(
        user=os.getenv("user"), password=os.getenv("password"),
        host=os.getenv("host"), port=os.getenv("port"), dbname=os.getenv("dbname")
    )

print("_conectar listo")

_conectar listo


In [8]:
def hacer_plan(pregunta: str) -> list[str]:
    """Devuelve una lista de pasos donde:
    - Paso 1: extracción de datos de la BD con SQL
    - Pasos 2+: operaciones sobre el DataFrame de pandas resultante
    """
    prompt = f"""Eres un experto en análisis de datos con SQL y pandas para Python.

Tienes la tabla "haciendas" en PostgreSQL con las columnas:
{COLUMNAS_VALIDAS_STR}

El usuario pregunta: "{pregunta}"

Genera un plan de resolución con estas reglas:
- El PRIMER paso debe describir qué datos extraer de la base de datos con SQL (una sola consulta amplia).
- Los pasos SIGUIENTES deben describir operaciones de transformación, filtrado o cálculo sobre el DataFrame de pandas resultante del paso anterior. NO deben consultar la base de datos.

Responde ÚNICAMENTE con un JSON array de strings, sin markdown. Ejemplo:
[
  "Extraer de la BD: Nombre_Unidad, Costo_Ha, Total_Cajas, FECHA para el año 2024",
  "Calcular el costo por caja dividiendo Costo_Ha entre Total_Cajas",
  "Ordenar de mayor a menor por costo por caja y quedarse con las 5 primeras"
]"""

    texto = _llm(prompt)
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
    return json.loads(texto.strip().rstrip("```").strip())

In [9]:
def generar_sql(actividad: str, pregunta_original: str) -> tuple[str, str]:
    """Paso 1: el LLM razona y genera el SQL de extracción contra la BD."""
    prompt = f"""Eres un experto en SQL para PostgreSQL y en el negocio bananero.

Tabla disponible:
{SCHEMA_CONTEXT}

Pregunta original del usuario: "{pregunta_original}"

Genera el SQL para extraer los datos necesarios para esta actividad:
"{actividad}"

Responde en este formato exacto:

RAZONAMIENTO:
<explica qué columnas extraes y por qué, en 2-4 oraciones>

SQL:
<solo el SQL, con saltos de línea e indentación>

Reglas del SQL:
- Tabla: "haciendas"
- Extrae columnas suficientes para que los pasos siguientes puedan operar sobre el DataFrame sin volver a la BD
- FECHA es DATE; para filtrar/agrupar por año: EXTRACT(YEAR FROM "FECHA")
- Comillas dobles en TODOS los nombres de columna
- Solo estas columnas (PROHIBIDO inventar otras): {COLUMNAS_VALIDAS_STR}
- Sin markdown, sin backticks"""

    texto = _llm(prompt)
    razonamiento, sql = "", ""
    if "SQL:" in texto:
        partes = texto.split("SQL:", 1)
        razonamiento = partes[0].replace("RAZONAMIENTO:", "").strip()
        sql = _limpiar_sql(partes[1])
    else:
        sql = _limpiar_sql(texto)
    return razonamiento, sql

In [10]:
def corregir_sql(sql: str) -> str:
    """Revisa el SQL y corrige cualquier nombre de columna que no exista en la tabla haciendas."""
    prompt = f"""Eres un experto en SQL para PostgreSQL.

Tienes la tabla "haciendas" con EXACTAMENTE estas columnas (respeta mayúsculas/minúsculas):
{COLUMNAS_VALIDAS_STR}

Revisa el siguiente SQL y corrige únicamente los nombres de columna que no correspondan a la lista anterior.
Si un nombre de columna no existe, reemplázalo por el nombre correcto más cercano de la lista.
No cambies nada más (lógica, filtros, alias de resultado, etc.).
Si el SQL ya es correcto, devuélvelo tal cual.

SQL a revisar:
{sql}

Devuelve SOLO el SQL corregido, sin explicaciones, sin markdown, sin backticks."""

    return _limpiar_sql(_llm(prompt))

In [11]:
def generar_pandas(actividad: str, pregunta_original: str, df: pd.DataFrame) -> tuple[str, str]:
    """Pasos 2+: el LLM razona y genera código pandas que opera sobre el DataFrame actual."""
    info_df = (
        f"Columnas: {list(df.columns)}\n"
        f"Tipos:    {df.dtypes.to_dict()}\n"
        f"Filas:    {len(df)}\n"
        f"Muestra:\n{df.head(5).to_string(index=False)}"
    )

    prompt = f"""Eres un experto en análisis de datos con pandas para Python.

Tienes un DataFrame llamado `df` con esta estructura:
{info_df}

Pregunta original del usuario: "{pregunta_original}"

Genera el código pandas para realizar esta operación sobre `df`:
"{actividad}"

Responde en este formato exacto:

RAZONAMIENTO:
<explica qué operación realizas y por qué, en 2-4 oraciones>

CODIGO:
<solo el código Python/pandas>

Reglas del código:
- El DataFrame de entrada se llama `df`
- El resultado final SIEMPRE debe quedar guardado en `df` como un pandas DataFrame (NO Series, NO lista, NO dict)
- Si usas .groupby(), termina con .reset_index() para que el resultado sea un DataFrame
- Si usas .value_counts() o cualquier método que devuelva una Series, conviértela a DataFrame con .reset_index()
- Solo usa pandas, no importes librerías adicionales
- Sin markdown, sin backticks"""

    texto = _llm(prompt)
    razonamiento, codigo = "", ""
    if "CODIGO:" in texto:
        partes = texto.split("CODIGO:", 1)
        razonamiento = partes[0].replace("RAZONAMIENTO:", "").strip()
        codigo = partes[1].strip()
        if codigo.startswith("```"):
            codigo = codigo.split("```")[1]
            if codigo.startswith("python"):
                codigo = codigo[6:]
        codigo = codigo.strip()
    else:
        codigo = texto.strip()
    return razonamiento, codigo

In [ ]:
def generar_plan(pregunta: str) -> list[str]:
    print(f"Pregunta: {pregunta}")
    print("=" * 60)

    pasos = hacer_plan(pregunta)

    print(f"Plan ({len(pasos)} actividades):")
    for i, p in enumerate(pasos, 1):
        print(f"  {i}. {p}")
    print()

    return pasos


"Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?"


Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?
Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?
¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?
¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?
¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?
¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?
¿Qué variables administrativas, pueden estar afectando los costos?
¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?
¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?
¿Qué programa se debería de ajustar para mejorar la productividad?

In [13]:
pregunta = "¿Cuáles son las 5 haciendas con mayor costo por hectárea en 2024?"

df_resultado = ejecutar_pipeline(pregunta)

Pregunta: ¿Cuáles son las 5 haciendas con mayor costo por hectárea en 2024?
Plan (5 actividades):
  1. Extraer de la BD: Nombre_Unidad, Costo_Ha, Total_Hectareas, FECHA para todos los registros donde el año de FECHA sea 2024
  2. Filtrar el DataFrame para asegurarse de que solo contiene registros del año 2024 usando pd.to_datetime en la columna FECHA
  3. Agrupar el DataFrame por Nombre_Unidad y calcular el promedio de Costo_Ha por hacienda usando groupby y mean
  4. Ordenar el DataFrame resultante de mayor a menor por el promedio de Costo_Ha usando sort_values con ascending=False
  5. Seleccionar las primeras 5 filas usando head(5) para obtener las 5 haciendas con mayor costo por hectárea

─── Paso 1 (SQL): Extraer de la BD: Nombre_Unidad, Costo_Ha, Total_Hectareas, FECHA para todos los registros donde el año de FECHA sea 2024
Razonamiento:
Se extraen las columnas "Nombre_Unidad", "Costo_Ha", "Total_Hectareas" y "FECHA" porque son las directamente necesarias para identificar cada haci

,Nombre_Unidad,Costo_Ha,Total_Hectareas,FECHA
0,Hacienda San Escriva,1129.323932,168.77,2024-01-01
1,Hacienda San Escriva,1129.110624,168.77,2024-02-01
2,Hacienda San Escriva,1073.093559,168.77,2024-03-01
3,Hacienda San Escriva,1143.437815,168.77,2024-04-01
4,Hacienda San Escriva,1324.850388,168.77,2024-05-01



─── Paso 2 (pandas): Filtrar el DataFrame para asegurarse de que solo contiene registros del año 2024 usando pd.to_datetime en la columna FECHA
Razonamiento:
Para filtrar solo los registros del año 2024, primero convertimos la columna FECHA a tipo datetime usando pd.to_datetime(), ya que actualmente es de tipo object. Luego extraemos el año con .dt.year y filtramos las filas donde el año sea igual a 2024. Esto garantiza que el análisis posterior solo considere datos del período solicitado.

Código:
df['FECHA'] = pd.to_datetime(df['FECHA'])
df = df[df['FECHA'].dt.year == 2024].reset_index(drop=True)

Resultado: 528 filas × 4 columnas


,Nombre_Unidad,Costo_Ha,Total_Hectareas,FECHA
0,Hacienda San Escriva,1129.323932,168.77,2024-01-01
1,Hacienda San Escriva,1129.110624,168.77,2024-02-01
2,Hacienda San Escriva,1073.093559,168.77,2024-03-01
3,Hacienda San Escriva,1143.437815,168.77,2024-04-01
4,Hacienda San Escriva,1324.850388,168.77,2024-05-01
...,...,...,...,...
523,Hacienda Recreo,1202.770465,79.77,2024-08-01
524,Hacienda Recreo,1024.457816,79.77,2024-09-01
525,Hacienda Recreo,1127.441394,79.77,2024-10-01
526,Hacienda Recreo,1095.988467,79.77,2024-11-01



─── Paso 3 (pandas): Agrupar el DataFrame por Nombre_Unidad y calcular el promedio de Costo_Ha por hacienda usando groupby y mean
Razonamiento:
Para identificar las 5 haciendas con mayor costo por hectárea en 2024, primero filtro el DataFrame para quedarse solo con registros del año 2024. Luego agrupo por `Nombre_Unidad` y calculo el promedio de `Costo_Ha` para cada hacienda, ya que cada una tiene múltiples registros mensuales. Finalmente ordeno de mayor a menor y tomo las primeras 5 filas.

Código:
df = (
    df[df['FECHA'].dt.year == 2024]
    .groupby('Nombre_Unidad', as_index=False)['Costo_Ha']
    .mean()
    .sort_values('Costo_Ha', ascending=False)
    .head(5)
    .reset_index(drop=True)
)

Resultado: 5 filas × 2 columnas


,Nombre_Unidad,Costo_Ha
0,Hacienda Union I,1848.244545
1,Hacienda San Carlos,1613.309074
2,Hacienda Calope,1588.092423
3,Hacienda Maravilla III,1284.517886
4,Hacienda Admiracion,1260.015608



─── Paso 4 (pandas): Ordenar el DataFrame resultante de mayor a menor por el promedio de Costo_Ha usando sort_values con ascending=False
Razonamiento:
El DataFrame ya contiene las 5 haciendas con mayor costo por hectárea en 2024, con una columna `Costo_Ha` que representa el promedio de costo. La operación consiste en ordenar este DataFrame de mayor a menor usando `sort_values` con `ascending=False` sobre la columna `Costo_Ha`, para presentar los resultados en orden descendente según el costo por hectárea.

Código:
df = df.sort_values(by='Costo_Ha', ascending=False).reset_index(drop=True)

Resultado: 5 filas × 2 columnas


,Nombre_Unidad,Costo_Ha
0,Hacienda Union I,1848.244545
1,Hacienda San Carlos,1613.309074
2,Hacienda Calope,1588.092423
3,Hacienda Maravilla III,1284.517886
4,Hacienda Admiracion,1260.015608



─── Paso 5 (pandas): Seleccionar las primeras 5 filas usando head(5) para obtener las 5 haciendas con mayor costo por hectárea
Razonamiento:
El DataFrame `df` ya contiene las 5 haciendas con mayor costo por hectárea, ordenadas de mayor a menor. La operación consiste en seleccionar las primeras 5 filas usando `head(5)` para obtener explícitamente ese subconjunto, confirmando el resultado solicitado por el usuario y asegurando que `df` quede como un DataFrame válido.

Código:
df = df.head(5)

Resultado: 5 filas × 2 columnas


,Nombre_Unidad,Costo_Ha
0,Hacienda Union I,1848.244545
1,Hacienda San Carlos,1613.309074
2,Hacienda Calope,1588.092423
3,Hacienda Maravilla III,1284.517886
4,Hacienda Admiracion,1260.015608


Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?
Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?
¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?
¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?
¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?
¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?
¿Qué variables administrativas, pueden estar afectando los costos?
¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?
¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?
¿Qué programa se debería de ajustar para mejorar la productividad?

In [14]:
preguntas = [
    "Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?",
    "Considerando las haciendas con las mismas variables (tamaño de la hacienda, certificaciones, numero de procesos, condición fitosanitaria, tipo de plagas, condiciones de suelo), ¿cuál es el ranking de haciendas en las diferentes categorías?",
    "¿Cuáles son las variables que más influyen en cada hacienda en sus niveles de costos?",
    "¿A qué se puede atribuir la tendencia actual en los indicadores de producción (merma, peso, productividad)?",
    "¿Qué debería de ajustarse a nivel de producción y costos, para reducir el costo las próximas semanas?",
    "¿Qué practicas tienen las haciendas con menor costo, que pueden ser replicadas en las demás haciendas?",
    "¿Qué variables administrativas, pueden estar afectando los costos?",      
    "¿Hay algún cambio en los programas de atención a la plantación que puedan estar afectando negativamente al costo?",  
    "¿Qué ha influido en la reducción de costos de las haciendas que han mejorado en los últimos 3 meses?",
    "¿Qué programa se debería de ajustar para mejorar la productividad?",

]

p = preguntas[0]
df_resultado = ejecutar_pipeline(p)

Pregunta: Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, ¿cuál sería el costo/caja y costo/hectárea real?
Plan (7 actividades):
  1. Extraer de la BD todas las columnas necesarias: FECHA, Semana, Zona, Unidad, Nombre_Unidad, Real, Total_Cajas, Total_Hectareas, Atencion_Plantacion, C_Riego, C_Mano_Obra_Riego, C_Mantenimiento_Riego, C_Combustible, C_Control_Sigatoca, C_Aplicacion_Aerea, C_Deshoje, C_Costos_Productos, C_Fertilizacion, C_Sacos_Fert, C_ManodeObra_Fert, C_Transporte_Fert, C_Administracion_Hacienda, Sueldos, Servicios_Basicos, C_Empaque_Fijo, Mantenimiento_Empacadora, Mantenimiento_Equipo, C_Logistica, Transporte, Materiales, Reclasificaciones_Transporte, Reclasificaciones_Materiales, C_Empaque_Variable, C_Cosecha, C_Transporte, C_Depreciaciones, Costo_Ha de toda la tabla haciendas sin filtros de fecha ni unidad
  2. Calcular el costo total por registro sumando todos los componentes de costo (Atencion_Plantacion, C_Riego, C_Mano_Obra_Riego, 

,FECHA,Semana,Zona,Unidad,Nombre_Unidad,Real,Costo_Ha,Atencion_Plantacion,C_Riego,C_Mano_Obra_Riego,...,Precipitacion_mm,Evotranspiracion,Humedad,Ausentismo_Agricola,Ausentismo_Justificado_Agricola,Ausentismo_Injustificado_Agricola,RotPerson_Salida_Todos_Motivos_Agricola,Pago_Labor_Persona,Pago_Por_Cuenta,Vacante_Labor
0,2020-01-01,5,Zona Camarones,2119,Hacienda San Jacinto,5.791250,789.755945,139352.0,3952,963,...,80.4,19.80,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2020-01-01,5,Zona Camarones,2120,Hacienda Lastenia,4.942464,867.574940,167813.0,5334,1403,...,88.4,13.74,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,2020-01-01,5,Zona Camarones,2121,Hacienda Wongkinmay,4.978442,732.552218,97469.0,5478,643,...,79.3,13.30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2020-01-01,5,Zona Camarones,2122,Hacienda Roblecito,5.421558,640.457552,87852.0,2914,643,...,92.2,13.80,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2020-01-01,5,Zona Camarones,2127,Hacienda San Simon,5.896796,886.093430,41852.0,499,771,...,115.8,10.40,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0



─── Paso 2 (pandas): Calcular el costo total por registro sumando todos los componentes de costo (Atencion_Plantacion, C_Riego, C_Mano_Obra_Riego, C_Mantenimiento_Riego, C_Combustible, C_Control_Sigatoca, C_Aplicacion_Aerea, C_Deshoje, C_Costos_Productos, C_Fertilizacion, C_Sacos_Fert, C_ManodeObra_Fert, C_Transporte_Fert, C_Administracion_Hacienda, Sueldos, Servicios_Basicos, C_Empaque_Fijo, Mantenimiento_Empacadora, Mantenimiento_Equipo, C_Logistica, Transporte, Materiales, Reclasificaciones_Transporte, Reclasificaciones_Materiales, C_Empaque_Variable, C_Cosecha, C_Transporte, C_Depreciaciones) en una nueva columna llamada Costo_Total_Calculado
Razonamiento:
Se suman todas las columnas de costo especificadas fila a fila usando el operador `sum` de pandas sobre el subconjunto de columnas correspondiente. Esto crea una nueva columna `Costo_Total_Calculado` en el DataFrame original que representa el costo total por registro, sumando cada componente de costo individual. Esta operación e

,FECHA,Semana,Zona,Unidad,Nombre_Unidad,Real,Costo_Ha,Atencion_Plantacion,C_Riego,C_Mano_Obra_Riego,...,Evotranspiracion,Humedad,Ausentismo_Agricola,Ausentismo_Justificado_Agricola,Ausentismo_Injustificado_Agricola,RotPerson_Salida_Todos_Motivos_Agricola,Pago_Labor_Persona,Pago_Por_Cuenta,Vacante_Labor,Costo_Total_Calculado
0,2020-01-01,5,Zona Camarones,2119,Hacienda San Jacinto,5.791250,789.755945,139352.0,3952,963,...,19.80,0.000,0.000000,0.000000,0.000000,0.0,0.00000,0.0,0,604490.0
1,2020-01-01,5,Zona Camarones,2120,Hacienda Lastenia,4.942464,867.574940,167813.0,5334,1403,...,13.74,0.000,0.000000,0.000000,0.000000,0.0,0.00000,0.0,0,702710.0
2,2020-01-01,5,Zona Camarones,2121,Hacienda Wongkinmay,4.978442,732.552218,97469.0,5478,643,...,13.30,0.000,0.000000,0.000000,0.000000,0.0,0.00000,0.0,0,432910.0
3,2020-01-01,5,Zona Camarones,2122,Hacienda Roblecito,5.421558,640.457552,87852.0,2914,643,...,13.80,0.000,0.000000,0.000000,0.000000,0.0,0.00000,0.0,0,402550.0
4,2020-01-01,5,Zona Camarones,2127,Hacienda San Simon,5.896796,886.093430,41852.0,499,771,...,10.40,0.000,0.000000,0.000000,0.000000,0.0,0.00000,0.0,0,183074.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2899,2025-06-01,27,Zona Vergel,2126,Hacienda Agrilechos II,NaN,829.966434,69357.0,1646,594,...,8.00,0.463,4.362745,2.450980,1.911765,0.0,34976.08834,0.0,3,268950.0
2900,2025-06-01,27,Zona Vergel,2128,Hacienda Libertad,NaN,886.502130,121823.0,2523,594,...,9.00,0.462,4.409449,2.992126,1.417323,0.0,58680.73526,0.0,8,485319.0
2901,2025-06-01,27,Zona Vergel,2129,Hacienda Mellizas,NaN,797.639857,66644.0,2667,594,...,10.00,0.401,3.703704,2.314815,1.388889,0.0,32602.78784,0.0,2,250553.0
2902,2025-06-01,27,Zona Vergel,2130,Hacienda Continental,NaN,1111.264530,172781.0,4671,950,...,12.00,0.425,5.035714,2.571429,2.464286,0.0,58177.93528,0.0,8,608532.0



─── Paso 3 (pandas): Filtrar únicamente los registros donde la columna Real sea igual a 0 o represente el valor presupuestado/programado (programa al 100%), identificando el código o etiqueta que distingue registros de programa versus registros de ejecución real en la columna Real
Razonamiento:
La columna `Real` parece contener valores que distinguen entre registros de programa (presupuestado) y registros de ejecución real. Analizando la muestra, los valores de `Real` rondan entre 4.9 y 5.9, lo que sugiere que son factores de cumplimiento o indicadores de rendimiento, no un flag binario 0/1. Para identificar qué valores representan "programa al 100%", primero exploraremos los valores únicos o distribución de `Real` para encontrar el patrón que distingue programa vs ejecución real, mostrando estadísticas descriptivas y valores únicos relevantes.

Código:
df_exploracion = df[['Real']].describe().reset_index()

df_valores_unicos = df['Real'].value_counts().reset_index()
df_valores_unicos

...

,Total_Cajas_sum,Total_Hectareas_sum,Costo_Total_sum,Cantidad_Registros,Costo_por_Caja,Costo_por_Hectarea,Costo_por_Caja_Programa,Costo_por_Hectarea_Programa
0,7.763230e+07,344055.740071,1.008910e+09,2376.0,12.996006,2932.40213,12.996006,2932.40213



─── Paso 6 (pandas): Calcular los valores globales (todas las haciendas consolidadas) sumando los totales de Costo_Total_Calculado, Total_Cajas y Total_Hectareas del DataFrame agrupado y dividiendo para obtener un Costo_por_Caja_Global y Costo_por_Hectarea_Global que representen el escenario de cumplimiento del 100% del programa en todas las haciendas
Razonamiento:
El DataFrame ya contiene los valores consolidados de todas las haciendas (1 sola fila con los totales sumados). Para calcular el costo/caja y costo/hectárea bajo un cumplimiento del 100% del programa, debemos usar las columnas `Costo_por_Caja_Programa` y `Costo_por_Hectarea_Programa` que ya representan ese escenario. Sin embargo, dado que el enunciado pide recalcular desde los totales, calcularemos `Costo_Total_sum` dividido entre `Total_Cajas_sum` y `Total_Hectareas_sum` usando los valores del programa, agregando las nuevas métricas globales como columnas adicionales al DataFrame.

Código:
costo_total_global = df['Costo_To

,Total_Cajas_sum,Total_Hectareas_sum,Costo_Total_sum,Cantidad_Registros,Costo_por_Caja,Costo_por_Hectarea,Costo_por_Caja_Programa,Costo_por_Hectarea_Programa,Costo_por_Caja_Global,Costo_por_Hectarea_Global
0,7.763230e+07,344055.740071,1.008910e+09,2376.0,12.996006,2932.40213,12.996006,2932.40213,12.996006,2932.40213



─── Paso 7 (pandas): Construir un DataFrame final que muestre por cada hacienda y a nivel global: Nombre_Unidad, Costo_Total_Programa, Total_Cajas_Programa, Total_Hectareas_Programa, Costo_por_Caja_Programa y Costo_por_Hectarea_Programa, ordenado de mayor a menor por Costo_por_Caja_Programa
Razonamiento:
El DataFrame `df` ya contiene una sola fila con los totales globales calculados (Costo_por_Caja_Programa y Costo_por_Hectarea_Programa coinciden con los valores globales). Dado que el DataFrame de entrada es un resumen agregado de una sola fila sin desglose por hacienda, construimos un DataFrame final con las columnas requeridas usando los datos disponibles, añadiendo una fila de nivel global con "Global" como nombre de unidad, ya que no hay información por hacienda en este contexto de entrada.

Código:
df_global = df[['Costo_Total_sum', 'Total_Cajas_sum', 'Total_Hectareas_sum', 'Costo_por_Caja_Programa', 'Costo_por_Hectarea_Programa']].copy()
df_global = df_global.rename(columns={
  

,Nombre_Unidad,Costo_Total_Programa,Total_Cajas_Programa,Total_Hectareas_Programa,Costo_por_Caja_Programa,Costo_por_Hectarea_Programa
0,Global,1.008910e+09,7.763230e+07,344055.740071,12.996006,2932.40213
